# AlphaFold2-Multimer refolding, Colab fallback

Runs the same predictions as `modal_app/af2_multimer.py`, on Colab's GPU, for
when Modal is unavailable or out of budget.

**Generated, not hand written.** The metrics code below is lifted verbatim from
the Modal app by `scripts/make_colab_notebook.py`, and a test fails if the two
diverge. Do not edit those cells; edit the source and regenerate.

## What this can and cannot do

Colab free gives a T4 with 16 GB. AlphaFold2-Multimer memory grows roughly with
the square of total length, so the largest complexes in this set will not fit.
The notebook measures the available memory, skips what cannot run, and records
the skip. It does not pretend to be a complete substitute for the Modal run.

Sessions disconnect without warning, so every job writes its result to Drive as
it completes, and re-running skips anything already done. Jobs run smallest
first to bank as many as possible before a disconnect.

## What you need in Drive

Put these under `MyDrive/interface_charge/`:

- `designs.csv`
- `test_set.csv`
- `interface_definitions.json` (from `scripts/01_define_interfaces.py`)

Results are written to `MyDrive/interface_charge/af2_results/`.

## 1. Check the GPU

Runtime, Change runtime type, T4 GPU.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 2. Install ColabFold

Takes a few minutes. Pins generated from the Modal image, so the two backends install the same stack.

In [ ]:
%%capture
# Pins generated from modal_app/af2_multimer.py. Two of them are not
# preferences and must not be raised:
#   colabfold 1.5.5 requires biopython<1.83 and numpy<2
#   dm-haiku 0.0.10 imports jax.linear_util, removed in jax 0.4.24, so
#   jax must stay at or below 0.4.23 or every prediction dies at import
!pip install -q "colabfold[alphafold]==1.5.5" "biopython<1.83" "numpy>=1.22,<2.0"
!pip install -q "jax[cuda12_pip]==0.4.23" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html

## 3. Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

BASE = Path("/content/drive/MyDrive/interface_charge")
RESULTS = BASE / "af2_results"
RESULTS.mkdir(parents=True, exist_ok=True)
(RESULTS / "msa_cache").mkdir(exist_ok=True)

for name in ("designs.csv", "test_set.csv", "interface_definitions.json"):
    path = BASE / name
    print(f"{'found  ' if path.is_file() else 'MISSING'} {path}")

## 4. Shared code

**Generated from `modal_app/af2_multimer.py`. Do not edit.**

In [ ]:
from typing import Any

import numpy as np


#: The AlphaFold parameter release. One constant: the weights downloaded into
#: the volume and the model requested at prediction time must be the same set,
#: and a mismatch is a wasted download followed by a failed run.
MODEL_TYPE: str = "alphafold2_multimer_v3"


#: Identifies this project to the free MMseqs2 server. ColabFold warns when it
#: is unset and says the warning will become an error, and it is basic courtesy
#: on a shared public resource.
MMSEQS_USER_AGENT: str = "interface-charge-rcsb/1.0 (AF2-Multimer charge study)"


def require_parseable_complex_a3m(a3m: str, chain_lengths: list[int]) -> None:
    """Refuse an alignment ColabFold would quietly reinterpret.

    ColabFold's ``unserialize_msa`` checks that the first line begins with ``#``
    and splits into exactly two tab-separated fields. If it does not, it does
    **not** raise: it falls through to a single-sequence branch and returns an
    alignment of depth one. Every downstream number then looks normal while the
    MSA has been discarded, which on this grid would mean paying for the whole
    run and reporting results the run did not actually produce.

    Silent reinterpretation is the failure mode worth spending code on, so the
    header is checked here against the same conditions, and against the chain
    lengths the alignment claims to describe.
    """
    lines = a3m.replace("\x00", "").splitlines()
    if len(lines) < 3:
        raise ValueError(
            f"the assembled a3m has {len(lines)} line(s); ColabFold requires at "
            "least three (header, query name, query sequence)."
        )
    header = lines[0]
    if not header.startswith("#"):
        raise ValueError(
            f"the a3m header is {header[:40]!r}. ColabFold expects it to start "
            "with '#', and silently treats anything else as a single sequence."
        )
    fields = header[1:].split("\t")
    if len(fields) != 2:
        raise ValueError(
            f"the a3m header splits into {len(fields)} tab-separated field(s), "
            "expected exactly two (lengths, cardinalities). ColabFold would "
            "discard the alignment rather than reject it."
        )
    declared = [int(v) for v in fields[0].split(",")]
    if declared != list(chain_lengths):
        raise ValueError(
            f"the a3m header declares chain lengths {declared} but the job's "
            f"chains are {list(chain_lengths)}. The alignment would be sliced "
            "against the wrong residues."
        )
    cardinality = [int(v) for v in fields[1].split(",")]
    if len(cardinality) != len(declared):
        raise ValueError(
            f"the a3m header declares {len(declared)} chain length(s) but "
            f"{len(cardinality)} cardinality value(s)."
        )


def _d0_scalar(length: float) -> float:
    """Yang and Skolnick (2004) length normalisation, reference scalar form.

    Reproduces ``calc_d0`` in the ipSAE reference, which returns exactly 1.0 at
    or below 27 residues rather than evaluating the cube root there. This
    differs from the array form below 28 residues, and the two are kept separate
    rather than unified because the reference applies each in a specific place
    and matching it is the point.
    """
    if length > 27.0:
        return max(1.0, 1.24 * (float(length) - 15.0) ** (1.0 / 3.0) - 1.8)
    return 1.0


def _d0_array(lengths: Any) -> Any:
    """Reference ``calc_d0_array``: floors the length at 26, then the result at 1.0."""
    import numpy as np

    clamped = np.maximum(26.0, np.asarray(lengths, dtype=float))
    return np.maximum(1.0, 1.24 * (clamped - 15.0) ** (1.0 / 3.0) - 1.8)


def ipsae(
    pae: Any,
    lengths: list[int],
    pae_cutoff_a: float,
) -> dict:
    """Interface pTM with the length normalisation taken from the interface.

    Why this is here at all: ipTM applies a d0 derived from the *total* number
    of residues in the complex. d0 grows with the cube root of that total, so
    the same physical interface scores differently depending on how large the
    chains around it happen to be. Across a set of complexes spanning a wide
    range of chain lengths, ipTM is therefore not comparable between complexes,
    and a charge effect estimated across them is confounded by size.

    ipSAE (Dunbrack, 2025) computes d0 from the number of residues actually
    involved in the interface instead, which removes the dependence on the parts
    of the chains that have nothing to do with binding.

    Implemented from the reference at github.com/DunbrackLab/IPSAE. Three
    normalisations are returned, all of them restricted to cross-chain pairs
    scoring below ``pae_cutoff_a``:

    ``ipsae_d0res``
        The headline score. d0 is recomputed for every aligned residue from the
        number of partner residues it confidently places.
    ``ipsae_d0dom``
        d0 from the count of distinct residues on both sides that participate in
        any confident pair, that is, from the size of the interface as a whole.
    ``ipsae_d0chn``
        d0 from the two chain lengths, so it differs from ipTM only by the
        cutoff. Reported as the control: if a charge trend appears in this one
        as strongly as in the others, the length normalisation was not what
        mattered.

    Each is asymmetric, so both directions are returned along with the maximum,
    which is the value the reference reports.
    """
    import numpy as np

    pae = np.asarray(pae, dtype=float)
    if len(lengths) != 2:
        raise ValueError(f"ipSAE is defined for a two-chain interface, got {len(lengths)} chains")
    total = int(sum(lengths))
    if pae.ndim != 2 or pae.shape != (total, total):
        raise ValueError(
            f"PAE matrix is {pae.shape}, expected ({total}, {total}) for chains of "
            f"lengths {lengths}. Refusing to score a matrix that does not match "
            "the sequences, because slicing it wrongly would silently score the "
            "wrong residue pairs."
        )

    first = int(lengths[0])
    chain_of = np.concatenate([np.zeros(first, dtype=int), np.ones(total - first, dtype=int)])

    result: dict = {"ipsae_pae_cutoff_a": float(pae_cutoff_a)}

    for direction, (aligned, scored) in enumerate([(0, 1), (1, 0)]):
        rows = chain_of == aligned
        # valid[i, j]: pair is cross-chain in this direction and confident.
        valid = np.outer(rows, chain_of == scored) & (pae < pae_cutoff_a)
        n_per_residue = valid.sum(axis=1)

        # d0res: one d0 per aligned residue, from its own partner count.
        d0_res = _d0_array(n_per_residue)
        with np.errstate(invalid="ignore", divide="ignore"):
            ptm_res = 1.0 / (1.0 + (pae / d0_res[:, None]) ** 2)

        # d0dom: one d0 for the pair, from the number of distinct residues on
        # either side that take part in any confident pair.
        n_interface = int((valid.any(axis=1) & rows).sum() + valid.any(axis=0).sum())
        d0_dom = _d0_scalar(n_interface)
        ptm_dom = 1.0 / (1.0 + (pae / d0_dom) ** 2)

        d0_chn = _d0_scalar(total)
        ptm_chn = 1.0 / (1.0 + (pae / d0_chn) ** 2)

        counts = np.where(n_per_residue > 0, n_per_residue, 1)
        by_residue = {
            "d0res": (ptm_res * valid).sum(axis=1) / counts,
            "d0dom": (ptm_dom * valid).sum(axis=1) / counts,
            "d0chn": (ptm_chn * valid).sum(axis=1) / counts,
        }
        label = f"{aligned}to{scored}"
        for name, values in by_residue.items():
            # Residues with no confident partner score zero, matching the
            # reference, rather than being dropped.
            scores = np.where(n_per_residue > 0, values, 0.0)[rows]
            result[f"ipsae_{name}_{label}"] = float(scores.max()) if scores.size else 0.0
        result[f"ipsae_n_interface_residues_{label}"] = n_interface
        result[f"ipsae_d0dom_value_{label}"] = d0_dom
        if direction == 0:
            result["ipsae_d0chn_value"] = d0_chn

    for name in ("d0res", "d0dom", "d0chn"):
        result[f"ipsae_{name}"] = max(result[f"ipsae_{name}_0to1"], result[f"ipsae_{name}_1to0"])
    return result


print('shared metrics code loaded')

## 5. Build the job list

Same pairing rules as the Modal launcher.

In [ ]:
import json

import pandas as pd

designs = pd.read_csv(BASE / "designs.csv")
test_set = pd.read_csv(BASE / "test_set.csv")
definitions = json.loads((BASE / "interface_definitions.json").read_text())

chains_by_id = dict(zip(test_set["pdb_id"], test_set["chains"]))

jobs = []
problems = []
for row in designs.itertuples():
    designed = str(row.designed_chain).split(",")[0].strip()
    pair = chains_by_id.get(row.pdb_id)
    entry = definitions.get(row.pdb_id)
    if pair is None or entry is None:
        problems.append(f"{row.pdb_id}: missing from test_set or definitions")
        continue
    partner = next((c.strip() for c in str(pair).split(",") if c.strip() != designed), None)
    partner_entry = entry["chains"].get(partner) if partner else None
    if partner_entry is None:
        problems.append(f"{row.pdb_id}: no partner sequence for chain {partner}")
        continue
    beta = f"{float(row.beta):+.4f}".replace("+", "p").replace("-", "m").replace(".", "_")
    jobs.append({
        "key": f"{row.pdb_id}_{designed}_beta{beta}_rep{int(row.replicate)}",
        "pdb_id": row.pdb_id,
        "beta": float(row.beta),
        "replicate": int(row.replicate),
        "designed_chain": designed,
        "partner_chain": partner,
        "chains": {designed: row.sequence, partner: partner_entry["sequence"]},
        "msa_mode": {designed: "single_sequence", partner: "msa"},
    })

# Smallest first. On Modal the largest go first so out-of-memory surfaces while
# abandoning is cheap; here the session may vanish at any moment, so the aim is
# to bank as many completed jobs as possible before it does.
jobs.sort(key=lambda j: sum(len(s) for s in j["chains"].values()))

print(f"{len(jobs)} job(s) over {len({j['pdb_id'] for j in jobs})} complex(es)")
if problems:
    print(f"{len(problems)} skipped while building the list:")
    for p in problems[:5]:
        print("  ", p)

## 6. Run

Resumable. Re-run this cell after a disconnect and it picks up where it stopped.

In [ ]:
import time
import traceback

import torch
from colabfold.batch import msa_to_str
from colabfold.batch import run as colabfold_run
from colabfold.colabfold import run_mmseqs2

# AlphaFold2-Multimer memory grows roughly with the square of total length. This
# cap is deliberately conservative: a job that dies takes the session with it,
# and a skipped job recorded honestly is worth more than a crashed notebook.
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
MAX_TOTAL_RESIDUES = 1000 if GPU_GB > 20 else 700
print(f"GPU has {GPU_GB:.0f} GB, capping jobs at {MAX_TOTAL_RESIDUES} total residues")


def build_mixed_a3m(job, ordered):
    """Native partner keeps its MSA, the design contributes depth one.

    Cached per complex and chain, not per job, because the partner sequence is
    identical at every beta and the MMseqs2 server is a free shared resource.
    """
    unpaired = []
    for chain_id in ordered:
        sequence = job["chains"][chain_id]
        if job["msa_mode"][chain_id] == "single_sequence":
            unpaired.append(f">{job['key']}_{chain_id}\n{sequence}\n")
            continue
        cached = RESULTS / "msa_cache" / f"{job['pdb_id']}_{chain_id}.a3m"
        if cached.is_file():
            unpaired.append(cached.read_text())
            continue
        result = run_mmseqs2(
            [sequence],
            str(RESULTS / "msa_cache" / f"mmseqs_{job['pdb_id']}_{chain_id}"),
            use_env=True, use_filter=True, use_templates=False, use_pairing=False,
            user_agent=MMSEQS_USER_AGENT,
        )
        lines = result[0] if isinstance(result, (list, tuple)) else result
        if not lines or ">" not in lines:
            raise RuntimeError(f"MMseqs2 returned no alignment for {job['pdb_id']} {chain_id}")
        cached.write_text(lines)
        unpaired.append(lines)
    # No paired block: pairing matches homologues by organism and a design has none.
    return msa_to_str(
        unpaired_msa=unpaired,
        paired_msa=None,
        query_seqs_unique=[job["chains"][c] for c in ordered],
        query_seqs_cardinality=[1] * len(ordered),
    )


done = skipped = failed = 0
for index, job in enumerate(jobs, start=1):
    out_dir = RESULTS / job["key"]
    metrics_path = out_dir / "metrics.json"
    if metrics_path.is_file():
        done += 1
        continue

    total = sum(len(s) for s in job["chains"].values())
    if total > MAX_TOTAL_RESIDUES:
        out_dir.mkdir(parents=True, exist_ok=True)
        metrics_path.write_text(json.dumps({
            "key": job["key"], "pdb_id": job["pdb_id"], "beta": job["beta"],
            "replicate": job["replicate"], "designed_chain": job["designed_chain"],
            "skipped": True,
            "reason": f"{total} residues exceeds the {MAX_TOTAL_RESIDUES} cap for a {GPU_GB:.0f} GB GPU",
        }, indent=2) + "\n")
        skipped += 1
        print(f"[{index}/{len(jobs)}] {job['key']}: SKIPPED, {total} residues")
        continue

    try:
        out_dir.mkdir(parents=True, exist_ok=True)
        started = time.time()
        ordered = sorted(job["chains"])
        query = ":".join(job["chains"][c] for c in ordered)
        lengths = [len(job["chains"][c]) for c in ordered]

        a3m = build_mixed_a3m(job, ordered)
        require_parseable_complex_a3m(a3m, lengths)

        colabfold_run(
            # A one-element LIST, not a string. ColabFold indexes a3m_lines[0],
            # so a bare string yields "#", fails the complex check, and silently
            # falls back to single sequence without raising.
            queries=[(job["key"], query, [a3m])],
            result_dir=str(out_dir),
            num_models=1,
            num_recycles=3,
            model_type=MODEL_TYPE,
            msa_mode="single_sequence",
            use_templates=False,
            random_seed=0,
            is_complex=True,
            rank_by="multimer",
            user_agent=MMSEQS_USER_AGENT,
        )

        scores_files = sorted(out_dir.glob("*scores*.json"))
        if not scores_files:
            raise FileNotFoundError("no ColabFold scores JSON; the prediction did not complete")
        scores = json.loads(scores_files[0].read_text())
        pae = np.array(scores.get("pae", []), dtype=float)

        metrics = {
            "key": job["key"], "pdb_id": job["pdb_id"], "beta": job["beta"],
            "replicate": job["replicate"], "designed_chain": job["designed_chain"],
            "complex_ptm": float(scores["ptm"]) if "ptm" in scores else None,
            "interface_ptm": float(scores["iptm"]) if "iptm" in scores else None,
            "mean_plddt": float(np.mean(scores["plddt"])) if scores.get("plddt") else None,
            "wall_clock_s": time.time() - started,
            "msa_paired": False,
            "source": "colab",
            "skipped": False,
        }
        if pae.size and pae.shape[0] == sum(lengths):
            first = lengths[0]
            metrics["interface_pae"] = float(
                (pae[:first, first:].mean() + pae[first:, :first].mean()) / 2.0
            )
            metrics.update(ipsae(pae, lengths, 10.0))

        metrics_path.write_text(json.dumps(metrics, indent=2, sort_keys=True, default=str) + "\n")
        done += 1
        print(f"[{index}/{len(jobs)}] {job['key']}: {metrics['wall_clock_s']/60:.1f} min, "
              f"ipSAE {metrics.get('ipsae_d0res', float('nan')):.3f}")
    except Exception:
        failed += 1
        print(f"[{index}/{len(jobs)}] {job['key']}: FAILED")
        traceback.print_exc()

print(f"\ndone {done}, skipped {skipped}, failed {failed}")

## 7. Collect

Writes one CSV you can download and hand to `scripts/04`.

In [ ]:
rows = [json.loads(p.read_text()) for p in sorted(RESULTS.glob("*/metrics.json"))]
table = pd.DataFrame(rows)
out = BASE / "af2_metrics_colab.csv"
table.to_csv(out, index=False)

ran = table[~table.get("skipped", False).astype(bool)] if "skipped" in table else table
print(f"{len(table)} record(s), {len(ran)} with predictions, written to {out}")
if len(ran):
    print(f"median wall clock: {ran['wall_clock_s'].median()/60:.1f} min")
    print(ran.groupby("beta")[["interface_ptm", "ipsae_d0res"]].median().round(3).to_string())
if "skipped" in table and table["skipped"].astype(bool).any():
    n = int(table["skipped"].astype(bool).sum())
    print(f"\n{n} job(s) skipped as too large for this GPU. They are recorded, not lost;")
    print("run them on Modal, or report the shortfall explicitly.")